# 🗂️ Notebook 2: Google Calendar — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Calendar** — owner + ACL.
- **Event** — base info + optional RRULE.
- **Occurrence** — a specific instance (derived from RRULE + base).
- **Reminder** — `{event_id, minutes_before, channel}`.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from datetime import datetime
from typing import Optional
from pydantic import BaseModel

class Event(BaseModel):
    id: int
    calendar_id: int
    title: str
    starts_at: datetime     # store UTC
    ends_at: datetime
    tz: str = "UTC"         # user's display tz
    rrule: Optional[str] = None   # e.g., "FREQ=WEEKLY;BYDAY=MO"

class Reminder(BaseModel):
    event_id: int
    minutes_before: int
    channel: str = "push"

from datetime import datetime, timezone
e = Event(id=1, calendar_id=1, title="1:1",
          starts_at=datetime(2026,5,1,15,tzinfo=timezone.utc),
          ends_at=datetime(2026,5,1,15,30,tzinfo=timezone.utc),
          tz="America/Los_Angeles",
          rrule="FREQ=WEEKLY;BYDAY=MO")
print(e)

## HTTP APIs

| Method | Path | What |
|---|---|---|
| POST | `/events` | Create |
| GET | `/events?from=..&to=..` | List occurrences in window |
| PATCH | `/events/{id}` | Update (this / this+future / series) |
| POST | `/events/{id}/invites` | Invite guests |


## Quick demo

In [ ]:
from datetime import datetime, timedelta, timezone

# UTC is the source of truth
start = datetime(2026, 3, 8, 15, tzinfo=timezone.utc)   # 15:00 UTC
# Display in Los Angeles — DST change!
try:
    from zoneinfo import ZoneInfo
    la = start.astimezone(ZoneInfo("America/Los_Angeles"))
    print("UTC:", start)
    print("LA (DST-aware):", la)
except Exception as e:
    print("zoneinfo unavailable:", e)

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.